# Шаг 6. KNN с учётом средних (KNNWithMeans, surprise + Optuna)

## Цель ноутбука
Обучить memory-based коллаборативную фильтрацию на основе KNNWithMeans,
подобрать гиперпараметры через Optuna (50 trials), оценить качество
и сравнить с baseline'ами и SVD из предыдущих шагов.

**Ключевое преимущество KNN перед SVD:** интерпретируемость —
для каждой рекомендации можно показать «похожие фильмы из вашей истории».

## 0. Импорты и настройки

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# ── Fallback-установка surprise / optuna ────────────────────────────────
try:
    import surprise  # noqa
except ImportError:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'scikit-surprise'], check=True)

try:
    import optuna  # noqa
except ImportError:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'optuna', 'plotly'], check=True)

import optuna
import optuna.visualization as ov
from surprise import KNNWithMeans, Dataset, Reader

from src.utils import SEED, set_seeds
from src.data_io import load_splits, load_features, load_id_maps
from src.metrics import (
    rmse, mae,
    evaluate_rating_prediction, evaluate_topn,
    build_ground_truth,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
optuna.logging.set_verbosity(optuna.logging.WARNING)
set_seeds()

MODELS_DIR    = Path('../models')
PROCESSED_DIR = Path('../data/processed')

print(f"SEED = {SEED}")
print(f"optuna version: {optuna.__version__}")

## 1. Загрузка данных

In [ ]:
splits          = load_splits()
train, val, test = splits['train'], splits['val'], splits['test']
features        = load_features()
movies_enriched = features['movies_enriched']
maps            = load_id_maps()

print(f"train: {len(train):,} | val: {len(val):,} | test: {len(test):,}")
print(f"Пользователей (train): {train['userId'].nunique()}")
print(f"Фильмов (train):       {train['movieId'].nunique()}")

## 2. Подготовка данных в формате Surprise

`KNNWithMeans` учитывает средние рейтинги пользователя или фильма (в зависимости от выбранного подхода)
и предсказывает отклонение от среднего как взвешенное среднее по соседям.
Это устраняет смещение из-за разной «строгости» пользователей.

In [ ]:
reader = Reader(rating_scale=(0.5, 5.0))


def df_to_surprise_trainset(df: pd.DataFrame):
    """Из DataFrame [userId, movieId, rating] построить Surprise Trainset."""
    data = Dataset.load_from_df(df[['userId', 'movieId', 'rating']], reader)
    return data.build_full_trainset()


trainset = df_to_surprise_trainset(train)
print(f'Surprise trainset: {trainset.n_users} users, '
      f'{trainset.n_items} items, {trainset.n_ratings} ratings')

## 3. Базовая KNN (точка отсчёта)

Item-based KNN с cosine similarity и k=40 — стандартная отправная точка.
Item-based обычно стабильнее user-based на малых датасетах:
похожих фильмов больше, чем похожих пользователей.

In [ ]:
baseline_sim_options = {'name': 'cosine', 'user_based': False}
baseline_knn = KNNWithMeans(k=40, min_k=1,
                            sim_options=baseline_sim_options,
                            verbose=False)

t0 = time.time()
baseline_knn.fit(trainset)
baseline_train_time = time.time() - t0
print(f'Baseline KNN обучена за {baseline_train_time:.2f} с')

# Оценка на val
val_preds_baseline = np.array([
    baseline_knn.predict(uid=u, iid=m).est
    for u, m in zip(val['userId'].values, val['movieId'].values)
])
baseline_val_metrics = evaluate_rating_prediction(val['rating'].values, val_preds_baseline)
print('Baseline KNN val:')
print(json.dumps(baseline_val_metrics, indent=2))

## 4. Optuna — подбор гиперпараметров

**Пространство параметров:**
- `user_based`: True/False — пользовательский или item-based подход
- `sim_name`: cosine / msd / pearson / pearson_baseline — мера схожести
- `k`: 10..100 — число соседей
- `min_k`: 1..5 — минимум соседей (если меньше → fallback на среднее)
- `min_support`: 1..10 — минимум общих оценок для расчёта схожести

**Цель:** минимизировать RMSE на val. 50 trials, TPE sampler.

In [ ]:
def objective(trial: optuna.Trial) -> float:
    user_based  = trial.suggest_categorical('user_based',
                                             [True, False])
    sim_name    = trial.suggest_categorical('sim_name',
                                             ['cosine', 'msd', 'pearson', 'pearson_baseline'])
    k           = trial.suggest_int('k', 10, 100)
    min_k       = trial.suggest_int('min_k', 1, 5)
    min_support = trial.suggest_int('min_support', 1, 10)

    sim_options = {
        'name':        sim_name,
        'user_based':  user_based,
        'min_support': min_support,
    }
    model = KNNWithMeans(k=k, min_k=min_k,
                         sim_options=sim_options, verbose=False)
    model.fit(trainset)
    preds = np.array([
        model.predict(uid=u, iid=m).est
        for u, m in zip(val['userId'].values, val['movieId'].values)
    ])
    return rmse(val['rating'].values, preds)


sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    study_name='knn_movielens'
)

print("Запускаем Optuna (50 trials)... Это займёт 10–30 минут.")
t0 = time.time()
study.optimize(objective, n_trials=50, show_progress_bar=True)
optuna_time = time.time() - t0

print(f'\nOptuna завершила {len(study.trials)} trials за {optuna_time:.1f} с')
print(f'Лучший RMSE на val: {study.best_value:.4f}')
print('Лучшие параметры:')
print(json.dumps(study.best_params, indent=2))

In [ ]:
# ── Визуализация Optuna ─────────────────────────────────────────────────

# 1. История оптимизации
fig_history = ov.plot_optimization_history(study)
fig_history.update_layout(title='Optuna: история оптимизации KNN (RMSE на val)')
fig_history.write_html(str(MODELS_DIR / 'optuna_knn_history.html'))
fig_history.show()

# 2. Важность параметров
fig_importance = ov.plot_param_importances(study)
fig_importance.update_layout(title='Optuna: важность гиперпараметров KNN')
fig_importance.write_html(str(MODELS_DIR / 'optuna_knn_importance.html'))
fig_importance.show()

# 3. Срезы по параметрам
fig_slice = ov.plot_slice(study)
fig_slice.update_layout(title='Optuna: срезы по гиперпараметрам KNN')
fig_slice.show()

# 4. Контурный график для пары k / min_k
fig_contour = ov.plot_contour(study, params=['k', 'min_k'])
fig_contour.update_layout(title='Optuna: контурный график k vs min_k')
fig_contour.show()

**Интерпретация результатов Optuna:**

- **user_based vs item_based:** на MovieLens-small item-based KNN обычно выигрывает,
  так как число фильмов (~9k) больше числа пользователей (~610),
  и матрица фильм–фильм более стабильна.
- **sim_name:** `pearson_baseline` часто даёт наилучший результат для предсказания рейтингов,
  поскольку учитывает baseline-смещения пользователей и фильмов.
  `msd` (Mean Squared Difference) иногда превосходит cosine на разреженных данных.
- **k:** оптимальное k обычно в диапазоне 20–60. Слишком малое k — высокая дисперсия,
  слишком большое k — усреднение нерелевантных соседей.
- **min_k:** значение 1–2 обычно оптимально; более высокое значение означает
  агрессивный fallback на среднее при недостатке соседей.

## 5. Финальная KNN на train + val

In [ ]:
best_params = study.best_params
final_sim_options = {
    'name':        best_params['sim_name'],
    'user_based':  best_params['user_based'],
    'min_support': best_params['min_support'],
}
final_knn = KNNWithMeans(
    k=best_params['k'],
    min_k=best_params['min_k'],
    sim_options=final_sim_options,
    verbose=False,
)

train_val     = pd.concat([train, val], ignore_index=True)
trainset_full = df_to_surprise_trainset(train_val)

t0 = time.time()
final_knn.fit(trainset_full)
final_train_time = time.time() - t0

print(f'Финальная KNN обучена на train+val за {final_train_time:.2f} с')
print(f'Параметры финальной модели: user_based={best_params["user_based"]}, '
      f'sim={best_params["sim_name"]}, k={best_params["k"]}, '
      f'min_k={best_params["min_k"]}, min_support={best_params["min_support"]}')

## 6. Оценка на test

### 6.1 RMSE / MAE

In [ ]:
test_preds = np.array([
    final_knn.predict(uid=u, iid=m).est
    for u, m in zip(test['userId'].values, test['movieId'].values)
])
knn_test_rating_metrics = evaluate_rating_prediction(test['rating'].values, test_preds)
print('KNN test (rating):')
print(json.dumps(knn_test_rating_metrics, indent=2))

### 6.2 Top-N метрики

Тот же протокол, что в шаге 5: для каждого пользователя из test ground_truth
берём всех непросмотренных кандидатов (из train+val), ранжируем предсказаниями,
берём топ-K. Это корректный протокол без утечки данных.

In [ ]:
def generate_topn_recommendations(model, user_ids, train_val_df,
                                   all_movies, k=20):
    """Ранжировать все непросмотренные фильмы для каждого пользователя и взять топ-K.

    model: объект Surprise с методом .predict(uid, iid).
    user_ids: список userId.
    train_val_df: DataFrame для определения уже просмотренных.
    all_movies: массив всех movieId (кандидатный пул).
    k: длина рекомендации.
    """
    seen_by_user = (
        train_val_df.groupby('userId')['movieId'].apply(set).to_dict()
    )
    all_movies = np.asarray(all_movies)
    recommendations = {}
    for user_id in user_ids:
        seen       = seen_by_user.get(user_id, set())
        candidates = all_movies[~np.isin(all_movies, list(seen))]
        scores     = np.array([model.predict(uid=user_id, iid=m).est
                               for m in candidates])
        top_k_idx  = np.argsort(-scores)[:k]
        recommendations[user_id] = candidates[top_k_idx].tolist()
    return recommendations


test_ground_truth = build_ground_truth(test, relevance_threshold=4.0)
test_users        = list(test_ground_truth.keys())
all_movies_arr    = train_val['movieId'].unique()

print(f'Генерация топ-20 для {len(test_users)} пользователей...')
t0 = time.time()
test_recs      = generate_topn_recommendations(
    final_knn, test_users, train_val, all_movies_arr, k=20
)
inference_time = time.time() - t0
print(f'Готово за {inference_time:.1f} с')

knn_test_topn_metrics = evaluate_topn(
    test_recs, test_ground_truth,
    ks=(5, 10, 20),
    all_items=all_movies_arr
)
print('KNN test (top-N):')
print(json.dumps(knn_test_topn_metrics, indent=2))

## 7. Сравнение с baseline и SVD

In [ ]:
with open(MODELS_DIR / 'popularity_metrics.json', 'r', encoding='utf-8') as f:
    pop_metrics = json.load(f)
with open(MODELS_DIR / 'svd_metrics.json', 'r', encoding='utf-8') as f:
    svd_metrics_loaded = json.load(f)

comparison_rows = [
    {
        'Модель':        'GlobalMean',
        'RMSE':          round(pop_metrics['global_mean']['test']['rmse'], 4),
        'MAE':           round(pop_metrics['global_mean']['test']['mae'], 4),
        'NDCG@10':       None,
        'Precision@10':  None,
        'Recall@10':     None,
        'HitRate@10':    None,
        'Coverage@20':   None,
    },
    {
        'Модель':        'Popularity',
        'RMSE':          None,
        'MAE':           None,
        'NDCG@10':       round(pop_metrics['popularity']['test']['ndcg@10'], 4),
        'Precision@10':  round(pop_metrics['popularity']['test']['precision@10'], 4),
        'Recall@10':     round(pop_metrics['popularity']['test']['recall@10'], 4),
        'HitRate@10':    round(pop_metrics['popularity']['test']['hit_rate@10'], 4),
        'Coverage@20':   round(pop_metrics['popularity']['test'].get('coverage@20', 0), 4),
    },
    {
        'Модель':        'SVD',
        'RMSE':          round(svd_metrics_loaded['final']['test_rating']['rmse'], 4),
        'MAE':           round(svd_metrics_loaded['final']['test_rating']['mae'], 4),
        'NDCG@10':       round(svd_metrics_loaded['final']['test_topn']['ndcg@10'], 4),
        'Precision@10':  round(svd_metrics_loaded['final']['test_topn']['precision@10'], 4),
        'Recall@10':     round(svd_metrics_loaded['final']['test_topn']['recall@10'], 4),
        'HitRate@10':    round(svd_metrics_loaded['final']['test_topn']['hit_rate@10'], 4),
        'Coverage@20':   round(svd_metrics_loaded['final']['test_topn'].get('coverage@20', 0), 4),
    },
    {
        'Модель':        'KNN',
        'RMSE':          round(knn_test_rating_metrics['rmse'], 4),
        'MAE':           round(knn_test_rating_metrics['mae'], 4),
        'NDCG@10':       round(knn_test_topn_metrics['ndcg@10'], 4),
        'Precision@10':  round(knn_test_topn_metrics['precision@10'], 4),
        'Recall@10':     round(knn_test_topn_metrics['recall@10'], 4),
        'HitRate@10':    round(knn_test_topn_metrics['hit_rate@10'], 4),
        'Coverage@20':   round(knn_test_topn_metrics.get('coverage@20', 0), 4),
    },
]
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.to_string(index=False))

# Визуализация — bar chart топ-N метрик
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

topn_metrics_names = ['NDCG@10', 'Precision@10', 'Recall@10', 'HitRate@10']
models_topn = ['Popularity', 'SVD', 'KNN']
colors      = ['darkorange', 'steelblue', 'seagreen']
x = np.arange(len(topn_metrics_names))
width = 0.25

for i, (model_name, color) in enumerate(zip(models_topn, colors)):
    row  = comparison_df[comparison_df['Модель'] == model_name].iloc[0]
    vals = [float(row[m]) if row[m] is not None else 0.0
            for m in topn_metrics_names]
    axes[0].bar(x + i * width, vals, width, label=model_name,
                color=color, alpha=0.85)

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(topn_metrics_names, fontsize=9)
axes[0].set_title('Top-N метрики: Popularity / SVD / KNN')
axes[0].legend()

rmse_models = ['GlobalMean', 'SVD', 'KNN']
rmse_colors = ['tomato', 'steelblue', 'seagreen']
rmse_vals = [
    pop_metrics['global_mean']['test']['rmse'],
    svd_metrics_loaded['final']['test_rating']['rmse'],
    knn_test_rating_metrics['rmse'],
]
axes[1].bar(rmse_models, rmse_vals, color=rmse_colors, alpha=0.85)
axes[1].set_title('RMSE на test')
axes[1].set_ylabel('RMSE')
for bar, v in zip(axes[1].patches, rmse_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.005,
                 f'{v:.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

**Интерпретация сравнительной таблицы:**

1. **KNN vs GlobalMean (RMSE):** KNN существенно лучше — за счёт учёта
   индивидуальных предпочтений через соседей.

2. **KNN vs SVD (RMSE):** SVD обычно немного точнее по RMSE, так как матричная
   факторизация лучше обобщается на разреженных данных. Это нормально и ожидаемо.

3. **KNN vs Popularity (NDCG@10):** KNN персонализирован и должен выигрывать,
   но при высокой разреженности (98%+) разрыв может быть небольшим.

4. **KNN vs SVD (NDCG@10):** результаты могут варьироваться в зависимости
   от датасета. KNN выигрывает за счёт точного соответствия "соседству",
   SVD — за счёт более плотного латентного представления.

## 8. Анализ интерпретируемости

**Главное преимущество KNN перед SVD — интерпретируемость.**
Для item-based KNN каждая рекомендация объясняется через похожие фильмы
из истории пользователя. Это ценно для доверия пользователя к системе.

In [ ]:
# Берём первого активного пользователя из test_users
sample_user = test_users[0]
sample_recs = test_recs[sample_user][:5]  # топ-5 рекомендаций

# История пользователя в train+val (высоко оценённые фильмы)
user_history = (
    train_val[
        (train_val['userId'] == sample_user) & (train_val['rating'] >= 4.0)
    ]
    .merge(movies_enriched[['movieId', 'title']], on='movieId')
    .sort_values('rating', ascending=False)
)

print(f'Пример пользователя: userId={sample_user}')
print(f'Высоко оценённые фильмы (rating >= 4.0, топ-10):')
display(user_history[['title', 'rating']].head(10))

print(f'\nТоп-5 рекомендаций KNN для userId={sample_user}:')
recs_with_titles = (
    movies_enriched[movies_enriched['movieId'].isin(sample_recs)]
    [['movieId', 'title']]
)
display(recs_with_titles)

In [ ]:
# ── Интерпретация через матрицу схожести (только item-based) ────────────
if not best_params['user_based']:
    sim_matrix = final_knn.sim   # shape (n_items_inner, n_items_inner)
    inner_id_of = trainset_full.to_inner_iid

    user_high_rated_movies = user_history['movieId'].tolist()
    explanations = []

    for rec_movie in sample_recs:
        try:
            inner_rec = inner_id_of(rec_movie)
        except ValueError:
            continue
        # Схожести между рекомендованным фильмом и каждым фильмом из истории
        sims = []
        for hist_movie in user_high_rated_movies:
            try:
                inner_hist = inner_id_of(hist_movie)
                sims.append((hist_movie, float(sim_matrix[inner_rec, inner_hist])))
            except ValueError:
                continue
        sims.sort(key=lambda x: x[1], reverse=True)
        top_similar = sims[:3]
        rec_title = movies_enriched.set_index('movieId').loc[rec_movie, 'title']
        for hist_movie, sim_val in top_similar:
            hist_title = movies_enriched.set_index('movieId').loc[hist_movie, 'title']
            explanations.append({
                'Рекомендовано':        rec_title,
                'Похожее из истории':   hist_title,
                'Схожесть':             round(sim_val, 4),
            })

    if explanations:
        print('\nОбъяснения рекомендаций через похожесть с историей пользователя:')
        display(pd.DataFrame(explanations))
    else:
        print('Не удалось извлечь объяснения (фильмы не найдены в trainset_full).')

else:
    print('Лучшая модель — user-based KNN.')
    print('Интерпретация: рекомендую X, потому что похожие на вас пользователи '
          'высоко оценили этот фильм.')
    # Для user-based: найти топ-3 похожих пользователей
    sim_matrix_u = final_knn.sim   # shape (n_users_inner, n_users_inner)
    inner_uid_of = trainset_full.to_inner_uid
    try:
        inner_sample = inner_uid_of(sample_user)
        user_sims    = sim_matrix_u[inner_sample]
        top_neighbor_inner = np.argsort(-user_sims)[1:4]  # skip self
        print(f'Топ-3 похожих пользователя (inner IDs): {top_neighbor_inner}')
        # Конвертируем inner → raw userId
        to_raw = trainset_full.to_raw_uid
        neighbor_raw_ids = [to_raw(i) for i in top_neighbor_inner]
        print(f'Топ-3 похожих пользователя (raw userId): {neighbor_raw_ids}')
        print('Их высоко оценённые фильмы формируют пул рекомендаций.')
    except Exception as e:
        print(f'Не удалось извлечь user-based объяснения: {e}')

## 9. Анализ ошибок

In [ ]:
# DataFrame с ошибками
test_with_errors = test.copy()
test_with_errors['pred']  = test_preds
test_with_errors['error'] = np.abs(test['rating'].values - test_preds)

# 1. Гистограмма ошибок и scatter истинный vs предсказанный
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(test_with_errors['error'], bins=30, color='steelblue',
             edgecolor='white', density=True)
axes[0].axvline(test_with_errors['error'].mean(), color='red',
                linestyle='--',
                label=f'Mean = {test_with_errors["error"].mean():.3f}')
axes[0].set_title('Распределение |ошибки| KNN на test')
axes[0].set_xlabel('|y_true - y_pred|')
axes[0].set_ylabel('Плотность')
axes[0].legend()

axes[1].scatter(test_with_errors['rating'], test_with_errors['pred'],
                alpha=0.15, s=5, color='steelblue')
lims = [0.5, 5.0]
axes[1].plot(lims, lims, 'r--', linewidth=1)
axes[1].set_xlim(lims); axes[1].set_ylim(lims)
axes[1].set_xlabel('Истинный рейтинг')
axes[1].set_ylabel('Предсказанный рейтинг')
axes[1].set_title('Истинный vs предсказанный (test)')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Топ-10 фильмов с наибольшей средней ошибкой
movie_error = (
    test_with_errors
    .groupby('movieId')['error']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'mean_error', 'count': 'n_test'})
)
movie_error = movie_error[movie_error['n_test'] >= 3]
top_movie_errors = (
    movie_error.nlargest(10, 'mean_error')
    .merge(movies_enriched[['movieId', 'title']], on='movieId', how='left')
)
top_movie_errors['mean_error'] = top_movie_errors['mean_error'].round(3)
print('Топ-10 фильмов с наибольшей средней |ошибкой| (min 3 оценки в test):')
display(top_movie_errors[['title', 'mean_error', 'n_test']])

In [ ]:
# 3. Топ-10 пользователей с наибольшей средней ошибкой
user_error = (
    test_with_errors
    .groupby('userId')['error']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'mean_error', 'count': 'n_test'})
)
user_error = user_error[user_error['n_test'] >= 3]
top_user_errors = user_error.nlargest(10, 'mean_error').copy()
top_user_errors['mean_error'] = top_user_errors['mean_error'].round(3)
print('Топ-10 пользователей с наибольшей средней |ошибкой|:')
display(top_user_errors)

In [ ]:
# 4. Корреляция ошибки с популярностью фильма в train
# (тест: есть ли у KNN bias на редкие фильмы из-за min_k fallback?)
movie_pop_train = (
    train.groupby('movieId').size().reset_index(name='n_ratings_train')
)
error_vs_pop = test_with_errors.merge(movie_pop_train, on='movieId', how='left')

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(error_vs_pop['n_ratings_train'], error_vs_pop['error'],
           alpha=0.1, s=5, color='steelblue')

from numpy.polynomial.polynomial import polyfit
x_vals = error_vs_pop['n_ratings_train'].values
y_vals = error_vs_pop['error'].values
mask   = ~np.isnan(x_vals) & ~np.isnan(y_vals)
coeffs = polyfit(x_vals[mask], y_vals[mask], 1)
x_line = np.linspace(x_vals[mask].min(), x_vals[mask].max(), 100)
ax.plot(x_line, coeffs[0] + coeffs[1] * x_line, 'r-', linewidth=2,
        label='тренд')

corr = np.corrcoef(x_vals[mask], y_vals[mask])[0, 1]
ax.set_xlabel('Число оценок фильма в train (популярность)')
ax.set_ylabel('|Ошибка| KNN')
ax.set_title(f'Ошибка KNN vs популярность фильма (Pearson r = {corr:.3f})')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Pearson r (ошибка vs популярность): {corr:.4f}')
print('Отрицательная корреляция → популярные фильмы предсказываются точнее.')
print('Позитивный bias на редких фильмах ожидаем: min_k fallback усредняет.')

**Выводы по анализу ошибок:**

1. **Распределение ошибок:** основная масса ошибок < 1.0 звезды, характерное
   для коллаборативной фильтрации на MovieLens.

2. **Фильмы с высокой ошибкой:** обычно это редкие или нишевые фильмы с малым
   числом оценок — именно здесь KNN переходит в `min_k` fallback.

3. **KNN и редкие фильмы:** если Pearson r отрицателен и значим — подтверждается
   гипотеза о том, что популярные фильмы предсказываются точнее.
   Для редких фильмов стоит применять popularity-фолбэк в продакшне.

4. **Сравнение с SVD:** SVD обычно лучше работает на редких фильмах, так как
   латентные факторы обобщают информацию глобально, а KNN опирается
   на локальное соседство.

## 10. Сохранение артефактов

In [ ]:
# Модель
joblib.dump(final_knn, MODELS_DIR / 'knn_model.pkl')
print(f"knn_model.pkl: {(MODELS_DIR / 'knn_model.pkl').stat().st_size / 1024 / 1024:.1f} MB")

# Параметры
knn_params = {
    'random_state':                SEED,
    'best_params':                 study.best_params,
    'optuna_n_trials':             50,
    'optuna_sampler':              'TPESampler',
    'optuna_direction':            'minimize',
    'optuna_target':               'rmse@val',
    'final_train_strategy':        'train + val concatenated',
    'baseline_train_time_sec':     baseline_train_time,
    'optuna_search_time_sec':      optuna_time,
    'final_train_time_sec':        final_train_time,
    'inference_time_test_topn_sec': inference_time,
}
with open(MODELS_DIR / 'knn_params.json', 'w', encoding='utf-8') as f:
    json.dump(knn_params, f, ensure_ascii=False, indent=2)

# Метрики
knn_metrics = {
    'baseline': {
        'val': baseline_val_metrics,
    },
    'final': {
        'val_best_rmse': float(study.best_value),
        'test_rating':   knn_test_rating_metrics,
        'test_topn':     knn_test_topn_metrics,
    },
    'meta': {
        'k_values':            [5, 10, 20],
        'relevance_threshold': 4.0,
        'optuna_n_trials':     50,
    },
}
with open(MODELS_DIR / 'knn_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(knn_metrics, f, ensure_ascii=False, indent=2)

# История trials Optuna
trials_df = study.trials_dataframe()
trials_df.to_parquet(MODELS_DIR / 'knn_optuna_trials.parquet', index=False)

print('Все артефакты сохранены.')
print(f'  Лучший RMSE (val): {study.best_value:.4f}')
print(f'  RMSE (test):       {knn_test_rating_metrics["rmse"]:.4f}')
print(f'  NDCG@10 (test):    {knn_test_topn_metrics["ndcg@10"]:.4f}')

## 11. Финальные проверки

In [ ]:
required_files = [
    MODELS_DIR / 'knn_model.pkl',
    MODELS_DIR / 'knn_params.json',
    MODELS_DIR / 'knn_metrics.json',
    MODELS_DIR / 'knn_optuna_trials.parquet',
    MODELS_DIR / 'optuna_knn_history.html',
    MODELS_DIR / 'optuna_knn_importance.html',
]
for p in required_files:
    assert p.exists(), f'Файл не найден: {p}'

# Round-trip модели
knn_loaded    = joblib.load(MODELS_DIR / 'knn_model.pkl')
sample_uid    = test['userId'].iloc[0]
sample_iid    = test['movieId'].iloc[0]
pred_loaded   = knn_loaded.predict(uid=sample_uid, iid=sample_iid).est
pred_original = final_knn.predict(uid=sample_uid, iid=sample_iid).est
assert abs(pred_loaded - pred_original) < 1e-9,     'Загруженная модель даёт другие предсказания'

# Sanity-границы
assert 0.5 < knn_test_rating_metrics['rmse'] < 1.5,     f"RMSE KNN подозрителен: {knn_test_rating_metrics['rmse']}"
assert 0.0 <= knn_test_topn_metrics['ndcg@10'] <= 1.0
assert 0.0 <= knn_test_topn_metrics['precision@10'] <= 1.0

# Ровно 50 trials
assert len(study.trials) == 50,     f'Optuna прогнала {len(study.trials)} trials вместо 50'

# Optuna улучшила baseline
assert study.best_value <= baseline_val_metrics['rmse'] + 1e-6,     'Optuna не улучшила RMSE baseline'

# KNN лучше GlobalMean по RMSE
gm_rmse = pop_metrics['global_mean']['test']['rmse']
assert knn_test_rating_metrics['rmse'] < gm_rmse,     f"KNN RMSE ({knn_test_rating_metrics['rmse']:.4f}) хуже GlobalMean ({gm_rmse:.4f})"

# Корректность рекомендаций
sample_user  = test_users[0]
sample_recs  = test_recs[sample_user]
assert len(sample_recs) == 20
assert len(set(sample_recs)) == 20, 'В рекомендациях есть дубликаты'
seen_by_user = set(train_val[train_val['userId'] == sample_user]['movieId'])
assert not (set(sample_recs) & seen_by_user),     'В рекомендациях KNN есть уже просмотренные фильмы'

# Метрики SVD доступны для сравнения
assert svd_metrics_loaded['final']['test_rating']['rmse'] > 0

print('\u2705 Шаг 6 пройден: KNN обучена, гиперпараметры подобраны, метрики сохранены')
print(f'   RMSE (test):    {knn_test_rating_metrics["rmse"]:.4f}  '
      f'(vs GlobalMean {gm_rmse:.4f}, '
      f'vs SVD {svd_metrics_loaded["final"]["test_rating"]["rmse"]:.4f})')
print(f'   NDCG@10 (test): {knn_test_topn_metrics["ndcg@10"]:.4f}  '
      f'(vs Popularity {pop_metrics["popularity"]["test"]["ndcg@10"]:.4f}, '
      f'vs SVD {svd_metrics_loaded["final"]["test_topn"]["ndcg@10"]:.4f})')